<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/7_cnns_for_text_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Convolutional Neural Networks for Text

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

A bag of words forgets word order: *"tax cut for schools"* and *"school cut for tax"* look the
same. Convolutional neural networks (**CNNs**) were invented for images, but in 2014 they turned
out to be excellent text classifiers too. The trick: slide a small window along the sentence and
learn which short word patterns (*income tax*, *public schools*, *health insurance*) matter.

In this notebook you will:

- see, step by step, what a convolution does to a sentence
- build a text CNN in **Keras** and in **PyTorch**
- start the network from the pre-trained **GloVe** vectors of Day 2
- open the model up and read off **which phrases each filter learned to detect**
- compare everything with the TF-IDF baseline and Day 3's BERT, on the same test set

**Contents**

1. The idea: a sliding window over words
2. The data
3. From text to sequences of numbers
4. A text CNN in Keras
5. A text CNN in PyTorch, with several window sizes
6. Starting from GloVe
7. What did the filters learn?
8. Comparing the models
9. Where CNNs stand today

> A GPU helps but isn't needed: everything here trains in a minute or two on Colab's CPU.

There are 6 exercises along the way, marked ✏️. A separate solutions notebook has worked answers, but try them yourself first.

## 0. Setup

Everything we need comes with Colab except `gensim` (for the GloVe vectors from Day 2), which the
first line installs. As in Bonus 6, we run Keras on the **PyTorch backend**, so both stacks share
one engine. The backend has to be set before Keras is imported.

In [ ]:
!pip install -q gensim

import os
os.environ["KERAS_BACKEND"] = "torch"     # must come before `import keras`

import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import keras

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
keras.utils.set_random_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 100)
print("Keras", keras.__version__, "on", keras.backend.backend(), "| PyTorch", torch.__version__,
      "| device:", device.upper())

## 1. The idea: a sliding window over words

Start from the word embeddings of Day 2: every word is a vector. A sentence then becomes a
*matrix*, with one row per word. A **filter** (also called a *kernel*) is a small matrix of
weights that covers a few consecutive words, say two or three. It slides along the sentence one
word at a time, and at each position computes a single number: how well the words under the
window match the pattern stored in its weights.

```
 the   bill   cuts   income   tax   rates
[────window────]                              → score
       [────window────]                       → score
              [────window────]                → score
                      ...
```

Let's do it by hand. To keep it readable, our "embeddings" have only two dimensions: how
*tax-related* a word is, and how *health-related*.

In [ ]:
words = ["the", "bill", "cuts", "income", "tax", "rates"]
emb = np.array([
    #  tax   health
    [0.0, 0.0],   # the
    [0.0, 0.0],   # bill
    [0.2, 0.1],   # cuts
    [0.9, 0.0],   # income
    [1.0, 0.0],   # tax
    [0.6, 0.0],   # rates
])

# A filter covering 2 words: it responds when BOTH words are tax-related
tax_filter = np.array([
    [1.0, -0.5],  # weights for the first word in the window
    [1.0, -0.5],  # weights for the second word
])

scores = []
for i in range(len(words) - 1):                       # slide over every pair of neighbors
    window = emb[i:i + 2]                             # 2 words x 2 dimensions
    scores.append(np.sum(window * tax_filter))        # multiply element-wise and add up

for i, s in enumerate(scores):
    print(f"{words[i]:>7} {words[i + 1]:<7} → {s:.2f}")
print(f"\nMax pooling keeps the strongest response: {max(scores):.2f} "
      f"(at '{words[int(np.argmax(scores))]} {words[int(np.argmax(scores)) + 1]}')")

Three things to take from this:

- A filter is a **pattern detector for short phrases**. This one fires on two tax-related words
  in a row. A width-3 filter is a learned *trigram* detector.
- **Max pooling** keeps only the strongest response across all positions. The question becomes
  "does this pattern appear *anywhere* in the title?", regardless of where.
- In a real network we don't design filters by hand. A layer has, say, 128 filters, all
  starting from random weights, and training shapes each one into a detector for phrases that
  help with the task. The 128 max-pooled scores then go to an ordinary output layer, as in
  Bonus 6.

Unlike a bag of n-grams, the filters work on *embeddings*, so a filter that learned *income tax*
also responds, somewhat, to *payroll tax* or *income levy*: similar words have similar vectors.

> **✏️ Exercise 1**
>
> 1. Write a **health** filter of width 2 for the toy example (it should respond to the second
>    dimension), and add the word pair `["health", "insurance"]` with embeddings `[0.0, 1.0]` and
>    `[0.1, 0.8]` to the end of the sentence. Where does your filter respond most strongly?
> 2. How many window positions does a width-3 filter have on a 6-word sentence? And on a sentence
>    of *n* words?

In [ ]:
# Your code here

## The data: US Congressional bills (again)

We use the same data and the **same split** as the Day 3 BERT notebook: bill titles from
`dreamproit/bill_labels_us`, four policy areas (Health, Education, Taxation, Armed Forces and
National Security), 500 bills per area, split 70/15/15 into training, validation and test sets
with the same random seed. That means you can put the scores from this notebook right next to
the TF-IDF baseline and the fine-tuned BERT model from Day 3.

(As on Day 3, we read only two ~200 MB chunks of the dataset's Parquet copy, and only the
columns we need, because the full dataset is a single 5 GB file.)

In [ ]:
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split

parts = []
for c in [0, 5]:
    path = hf_hub_download(repo_id="dreamproit/bill_labels_us", repo_type="dataset",
                           revision="refs/convert/parquet", filename=f"default/train/{c:04d}.parquet")
    parts.append(pd.read_parquet(path, columns=["title", "policy_area"]))
raw = pd.concat(parts, ignore_index=True)

AREAS = ["Health", "Education", "Taxation", "Armed Forces and National Security"]
clean = (raw.assign(title=raw["title"].fillna("").str.strip())
            .query("title != ''")
            .drop_duplicates(subset="title"))                 # every title once: no leakage
bills = (clean[clean["policy_area"].isin(AREAS)]
         .sample(frac=1, random_state=SEED)
         .groupby("policy_area").head(500)
         .reset_index(drop=True))

labels = sorted(AREAS)
label2id = {name: i for i, name in enumerate(labels)}
bills["label"] = bills["policy_area"].map(label2id)

train_df, temp_df = train_test_split(bills, test_size=0.30, stratify=bills["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)
y_train, y_val, y_test = train_df["label"].values, val_df["label"].values, test_df["label"].values

print(f"train: {len(train_df)}   validation: {len(val_df)}   test: {len(test_df)}")
print("Label IDs:", label2id)

## From text to sequences of numbers

Convolutional and recurrent networks read text **in order**, so we can't use TF-IDF (which
throws the order away). Instead, every title becomes a sequence of word IDs:

1. **Tokenize**: lowercase, split into words.
2. **Build a vocabulary** from the *training* titles only. Words that appear only once are
   mapped to a shared "unknown" token, because the model can't learn anything useful from a
   single example (and so that validation and test words the model never saw are handled the
   same way).
3. **Encode and pad**: replace every word by its ID, and cut or pad every title to the same
   length, so titles can be processed in batches.

This step is the same for both stacks, so we do it once, in plain Python and NumPy.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

PAD, UNK = 0, 1                                   # two reserved IDs
counts = Counter(w for t in train_df["title"] for w in tokenize(t))
vocab = ["<pad>", "<unk>"] + sorted(w for w, c in counts.items() if c >= 2)
word2id = {w: i for i, w in enumerate(vocab)}
print(f"Vocabulary: {len(vocab):,} words (from {len(counts):,} distinct words in the training titles)")

lengths = train_df["title"].apply(lambda t: len(tokenize(t)))
MAX_LEN = int(np.percentile(lengths, 95))         # long enough for 95% of titles
print(f"Title length in words: median {int(lengths.median())}, 95th percentile {MAX_LEN} -> MAX_LEN = {MAX_LEN}")

def encode(texts, max_len=MAX_LEN):
    out = np.full((len(texts), max_len), PAD, dtype="int64")
    for row, text in enumerate(texts):
        ids = [word2id.get(w, UNK) for w in tokenize(text)][:max_len]   # truncate long titles
        out[row, :len(ids)] = ids                                      # pad short ones with 0s
    return out

X_train, X_val, X_test = encode(train_df["title"]), encode(val_df["title"]), encode(test_df["title"])

example = train_df["title"].iloc[0]
print("\nExample title:", example)
print("Encoded:      ", X_train[0])
print("Decoded back: ", " ".join(vocab[i] for i in X_train[0] if i != PAD))

> **✏️ Exercise 2**
>
> We picked `MAX_LEN` so that 95% of titles fit. Plot a histogram of the title lengths (in words)
> with a vertical line at `MAX_LEN`. What share of the *training* titles gets truncated? Why might
> truncation matter less for a CNN with max pooling than you'd think?

In [ ]:
# Your code here

### The baseline to beat

In [ ]:
results = {}

def score(name, y_true, y_pred):
    results[name] = {"accuracy": accuracy_score(y_true, y_pred),
                     "macro-F1": f1_score(y_true, y_pred, average="macro")}
    print(f"{name:<34} validation accuracy {results[name]['accuracy']:.3f} | "
          f"macro-F1 {results[name]['macro-F1']:.3f}")

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
baseline = LogisticRegression(max_iter=1000).fit(tfidf.fit_transform(train_df["title"]), y_train)
score("TF-IDF + logistic regression", y_val, baseline.predict(tfidf.transform(val_df["title"])))

## 4. A text CNN in Keras

The network, layer by layer:

1. **Embedding**: turns every word ID into a 50-dimensional vector (learned from scratch for now).
2. **Conv1D**: 128 filters, each 3 words wide, with a ReLU. *1D* because the window moves in one
   direction, along the sentence.
3. **GlobalMaxPooling1D**: for each filter, keep its strongest response anywhere in the title.
4. **Dropout** and a **Dense** output layer with one probability per policy area.

In [ ]:
EMB_DIM = 50          # same size as the GloVe vectors we'll use later, for a fair comparison
N_FILTERS = 128
n_classes = len(labels)

keras.utils.set_random_seed(SEED)
keras_cnn = keras.Sequential([
    keras.Input(shape=(MAX_LEN,), dtype="int64"),
    keras.layers.Embedding(input_dim=len(vocab), output_dim=EMB_DIM),
    keras.layers.Conv1D(filters=N_FILTERS, kernel_size=3, activation="relu"),
    keras.layers.GlobalMaxPooling1D(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(n_classes, activation="softmax"),
])
keras_cnn.summary()

Most parameters sit in the embedding layer (one 50-number vector per vocabulary word). The
convolution layer has `3 words × 50 dimensions × 128 filters` weights plus 128 biases.

In [ ]:
keras_cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
start = time.time()
keras_history = keras_cnn.fit(X_train, y_train, validation_data=(X_val, y_val),
                              epochs=15, batch_size=32, callbacks=[early_stop], verbose=0)
print(f"Trained {len(keras_history.history['loss'])} epochs in {time.time() - start:.1f} s")

score("Keras CNN (width 3)", y_val, keras_cnn.predict(X_val, verbose=0).argmax(axis=1))

In [ ]:
def plot_history(train_loss, val_loss, title):
    plt.figure(figsize=(6.5, 3.2))
    plt.plot(train_loss, label="training loss", color="#1A1A2E")
    plt.plot(val_loss, label="validation loss", color="#34B233")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(title); plt.legend()
    plt.show()

plot_history(keras_history.history["loss"], keras_history.history["val_loss"], "Keras CNN")

> **✏️ Exercise 3**
>
> Build two variations of the Keras CNN and compare their validation macro-F1 with the original:
>
> - wider windows: `kernel_size=5`
> - fewer filters: `filters=32`
>
> Which change matters more here? What would you expect for longer texts, such as speeches?

In [ ]:
# Your code here

## 5. A text CNN in PyTorch, with several window sizes

In PyTorch we can easily go one step further and use the design from Kim (2014), the paper that
made CNNs popular for text: **several window sizes side by side**. Here, 64 filters each of
widths 2, 3 and 4 look at the same title in parallel, and their max-pooled scores are
concatenated before the output layer.

Two PyTorch details: `nn.Conv1d` expects the embedding dimensions as *channels*, so we swap two
axes with `transpose`; and `padding_idx=0` keeps the padding token's vector fixed at zero.

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, n_classes, widths=(2, 3, 4), n_filters=64,
                 dropout=0.5, pretrained=None, freeze=False):
        super().__init__()
        if pretrained is None:
            self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        else:                                             # start from given vectors (Section 6)
            self.embedding = nn.Embedding.from_pretrained(torch.tensor(pretrained),
                                                          freeze=freeze, padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, n_filters, kernel_size=w) for w in widths])
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(n_filters * len(widths), n_classes)

    def forward(self, x):                                 # x: [batch, MAX_LEN] word IDs
        e = self.embedding(x).transpose(1, 2)             # -> [batch, emb_dim, MAX_LEN]
        pooled = [torch.relu(conv(e)).max(dim=2).values   # convolve, ReLU, max over positions
                  for conv in self.convs]                 # one [batch, n_filters] per width
        return self.out(self.dropout(torch.cat(pooled, dim=1)))   # logits


torch.manual_seed(SEED)
torch_cnn = TextCNN(len(vocab), EMB_DIM, n_classes).to(device)
print(torch_cnn)
print("Parameters:", sum(p.numel() for p in torch_cnn.parameters()))

The training loop is the same as in Bonus 6, with early stopping built in (Exercise 5 of Bonus 6).

In [ ]:
def make_loader(X, y, shuffle, batch_size=32):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)


def predict_torch(model, loader):
    model.eval()
    preds, total_loss = [], 0.0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            total_loss += nn.functional.cross_entropy(logits, yb.to(device)).item() * len(yb)
            preds.append(logits.argmax(dim=1).cpu())
    return torch.cat(preds).numpy(), total_loss / len(loader.dataset)


def train_torch(model, epochs=15, lr=1e-3, patience=3):
    # Standard training loop with early stopping on the validation loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best, best_state, bad_epochs, history = float("inf"), None, 0, {"loss": [], "val_loss": []}
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item() * len(yb)
        _, val_loss = predict_torch(model, val_loader)
        history["loss"].append(running / len(train_loader.dataset))
        history["val_loss"].append(val_loss)
        if val_loss < best:
            best, best_state, bad_epochs = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break
    model.load_state_dict(best_state)
    print(f"Trained {len(history['loss'])} epochs; best validation loss {best:.4f}")
    return history

In [ ]:
start = time.time()
torch_history = train_torch(torch_cnn)
print(f"... in {time.time() - start:.1f} s")
score("PyTorch CNN (widths 2-4)", y_val, predict_torch(torch_cnn, val_loader)[0])
plot_history(torch_history["loss"], torch_history["val_loss"], "PyTorch CNN")

> **✏️ Exercise 4**
>
> Keras can build multi-width networks too, but not with `Sequential`, which only stacks layers
> in a single line. You need the **functional API**, where you connect layers like function
> calls. Build the Kim-style network in Keras: one `Embedding`, then three parallel branches
> (`Conv1D` with widths 2, 3 and 4, each followed by `GlobalMaxPooling1D`), joined with
> `keras.layers.Concatenate()`, then dropout and the output layer. Train and compare.

In [ ]:
# Your code here

## 6. Starting from GloVe

So far the embedding layer started from random numbers and had to learn word meanings from just
1,400 titles. We can give it a head start with the **GloVe** vectors from Day 2, which were
learned from billions of words. This is *transfer learning* in its simplest form, the same idea
that BERT takes much further (Day 3).

In [ ]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-50")      # ~66 MB, the same vectors as on Day 2
EMB_GLOVE = glove.vector_size

# One row per vocabulary word: the GloVe vector if GloVe knows the word, small random numbers if not
rng = np.random.default_rng(SEED)
embedding_matrix = rng.normal(0, 0.1, (len(vocab), EMB_GLOVE)).astype("float32")
embedding_matrix[PAD] = 0.0
found = 0
for word, idx in word2id.items():
    if word in glove.key_to_index:
        embedding_matrix[idx] = glove[word]
        found += 1
print(f"GloVe covers {found:,} of {len(vocab):,} vocabulary words ({found / len(vocab):.0%})")

Now we train the same PyTorch CNN, once starting from GloVe and letting training adjust the
vectors (`freeze=False`, *fine-tuning* the embeddings).

In [ ]:
torch.manual_seed(SEED)
glove_cnn = TextCNN(len(vocab), EMB_GLOVE, n_classes, pretrained=embedding_matrix, freeze=False).to(device)
train_torch(glove_cnn)
score("PyTorch CNN + GloVe (fine-tuned)", y_val, predict_torch(glove_cnn, val_loader)[0])

With small training sets, pre-trained embeddings usually help, sometimes a lot, because the model
already knows that *Medicare* and *Medicaid* are related before it has seen many examples of
either. With plenty of training data the gap shrinks.

> **✏️ Exercise 5**
>
> 1. Train a third version with **frozen** GloVe embeddings (`freeze=True`): the vectors stay
>    exactly as GloVe learned them, and only the convolution and output layers are trained. How
>    does it compare with the fine-tuned version and with random embeddings?
> 2. In Keras, you can start an `Embedding` layer from the same matrix with
>    `embeddings_initializer=keras.initializers.Constant(embedding_matrix)` and freeze it with
>    `trainable=False`. Build and train the frozen GloVe version of the width-3 Keras CNN.

In [ ]:
# Your code here

## 7. What did the filters learn?

Here is one of the nicest properties of text CNNs: you can **read** what each filter detects. For
a given filter, run it over every training title, and list the word windows where it responds
most strongly. We look at the width-3 filters of the GloVe CNN.

In [ ]:
def top_phrases(model, filter_idx, width=3, k=6):
    # The k word windows (from the training titles) where one filter responds most strongly
    width_pos = [conv.kernel_size[0] for conv in model.convs].index(width)
    conv = model.convs[width_pos]
    model.eval()
    with torch.no_grad():
        e = model.embedding(torch.tensor(X_train).to(device)).transpose(1, 2)
        act = torch.relu(conv(e))[:, filter_idx, :].cpu().numpy()   # [n_titles, n_positions]

    best = []
    for row, pos in zip(*np.unravel_index(np.argsort(act, axis=None)[::-1], act.shape)):
        ids = X_train[row, pos:pos + width]
        if PAD in ids:                                   # skip windows that run into padding
            continue
        phrase = " ".join(vocab[i] for i in ids)
        if phrase not in [p for p, _ in best]:
            best.append((phrase, act[row, pos]))
        if len(best) == k:
            break
    return best


for f in range(6):
    phrases = top_phrases(glove_cnn, f)
    print(f"Filter {f}: " + " | ".join(p for p, _ in phrases))

Some filters latch onto clear topic phrases; others pick up formulaic legal language (*"to amend
the"*, *"and for other"*) or look like noise. That mix is typical, and reading filters like this is
a useful sanity check: if the strongest phrases had nothing to do with policy content (say,
bill numbers or dates), the model would be learning a shortcut.

> **Caution:** as with attention maps on Day 3, this is a window into the model, not a full
> explanation of its decisions. The output layer combines all filters, and one filter's
> favorite phrases don't tell you how much it counts.

> **✏️ Exercise 6**
>
> Look at the top phrases of filters 6 to 20 of the GloVe CNN. Pick three filters and give each a
> short label, as you would when interpreting topics in a topic model (e.g., "veterans' health
> care", "tax credits"). How many filters are easy to label? What does that tell you about using
> filters as a measurement instrument?

In [ ]:
# Your code here

## 8. Comparing the models

All validation scores so far:

In [ ]:
pd.DataFrame(results).T.round(3)

And the test set, once, for the final models. These are the same 300 titles as in the Day 3
notebook, so you can compare directly with its TF-IDF baseline and fine-tuned DistilBERT.

In [ ]:
test_preds = {
    "TF-IDF + logistic regression": baseline.predict(tfidf.transform(test_df["title"])),
    "Keras CNN (width 3)": keras_cnn.predict(X_test, verbose=0).argmax(axis=1),
    "PyTorch CNN (widths 2-4)": predict_torch(torch_cnn, test_loader)[0],
    "PyTorch CNN + GloVe": predict_torch(glove_cnn, test_loader)[0],
}
pd.DataFrame({name: {"test accuracy": accuracy_score(y_test, p),
                     "test macro-F1": f1_score(y_test, p, average="macro")}
              for name, p in test_preds.items()}).T.round(3)

## 9. Where CNNs stand today

Between roughly 2014 and 2018, CNNs were among the best text classifiers available. Then
pre-trained transformers like BERT (Day 3) overtook them on almost every benchmark, because they
read the whole text at once with attention and arrive with knowledge from huge amounts of text.

CNNs still have their place:

- They are **small and fast**: they train in seconds and run comfortably on a laptop, which
  matters when you classify millions of social media posts.
- On **short texts with clear keywords** (titles, headlines, tweets), they're often close to a
  transformer.
- Their filters are **easier to inspect** than a transformer's attention heads.

And the idea of convolution is far from dead: it remains the workhorse for **images** (Bonus 9).

### Further reading

- Kim, Y. (2014). Convolutional neural networks for sentence classification. *Proceedings of
  EMNLP 2014*. The paper behind Section 5.
- Zhang, Y., & Wallace, B. (2017). A sensitivity analysis of (and practitioners' guide to)
  convolutional neural networks for sentence classification. *Proceedings of IJCNLP 2017*.
  Practical advice on window sizes, filters and regularization.
- Goldberg, Y. (2017). *Neural network methods for natural language processing*. Morgan & Claypool.